# Sinhala corpus: production pilot with Surya

Run after merging these changes into `dev`, or set `BRANCH` to a pushed feature branch containing this notebook. Choose a **T4 GPU** runtime.
This notebook collects a small public PDF, processes selected pages and prepares a
review packet on Drive. It never approves OCR text or exports training data automatically.

The first run creates a **new deployment store**, preserving the seed catalog but
resetting its download states because laptop files have not been copied to Drive.
Existing stores and raw files are preserved on subsequent runs. Keep one runtime/writer
per store. This is separate from `reviews/ocr/broad_20261011` and its OCR evaluation files.

Start with Biology guide pages 12–13, two pages total. Review every word and equation
before expanding. Read `docs/CORPUS_PRODUCTION_GUIDE.md` for limits and approvals.


In [ ]:
from pathlib import Path
import json, subprocess, sys, shutil, datetime
from google.colab import drive

if not shutil.which('nvidia-smi'):
    raise RuntimeError('Choose Runtime → Change runtime type → T4 GPU, then reconnect.')
subprocess.run(['nvidia-smi'], check=True)
drive.mount('/content/drive')
ROOT = Path('/content/sinhala-corpus')
STORE = Path('/content/drive/MyDrive/sinhala-corpus/production_v1')
BRANCH = 'dev'
DOC_IDS = ['LK-EDU-000264']
PAGE_NUMBERS = [12, 13]  # Set None later to process the whole selected PDF in batches.
MAX_PAGES = 2
if ROOT.exists():
    raise RuntimeError('Repository already exists. Use a fresh runtime for setup; resume processing using the later cells.')
subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',
                'https://github.com/Warusha-Shamalka/sinhala-corpus.git',str(ROOT)],check=True)


## Isolated pinned environment

Packages and model weights are downloaded during setup. OCR runs from local weights
in a separate process with network access blocked. Temporary models live in `/content`;
raw documents, processed pages, review packets and audit manifests live on Drive.


In [ ]:
ENV = Path('/content/corpus-env')
if ENV.exists():
    raise RuntimeError('Use a fresh runtime instead of replacing an existing environment.')
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
subprocess.run([sys.executable,'-m','uv','venv','--python','3.12.12','--seed',str(ENV)],check=True)
PYTHON = str(ENV/'bin/python')
def pip(*args):
    subprocess.run([PYTHON,'-m','pip','install',*args],check=True)
pip('torch==2.7.1','torchvision==0.22.1','--index-url','https://download.pytorch.org/whl/cu126')
pip('PyYAML==6.0.3','surya-ocr==0.17.1','transformers==4.57.6','Pillow==10.4.0')
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','poppler-utils'],check=True)


## Persistent store

This initialization is for a fresh store. It archives the repository seed catalog and
creates a separate acquisition catalog. It does not migrate old raw files or alter the
repository catalog. If `production_v1` already exists, its existing catalog is used.
Do not delete the store to resume a run.


In [ ]:
CONFIG = STORE/'configs/pipeline.yaml'
if not STORE.exists():
    subprocess.run([PYTHON,'-B','-m','pipeline.workflow','init','--store',str(STORE)],cwd=ROOT,check=True)
if not CONFIG.is_file():
    raise RuntimeError('Store exists without its configuration. Inspect it; choose a new store only for a new deployment.')


## Freeze Surya checkpoints and run environment

Surya does not require a Hugging Face token. Its pinned detector and recognizer are
verified by SHA256. Changing an OCR backend/revision creates separate page artifacts;
see the guide before installing a different backend's dependencies.


In [ ]:
MODEL_PATH = Path('/content/surya-models')
selection_code = r"""
import json,sys
from pathlib import Path
from pipeline.ocr.surya_backend import prepare_models,verify_models
from pipeline.download.downloader import file_hash
root=Path(sys.argv[1]); output=Path(sys.argv[2])
if root.exists():
    revision=file_hash(root/'selection.json')
    result={**verify_models(root,revision),'path':str(root),'revision':revision,
            'revision_kind':'local_components_manifest_sha256'}
else:
    result=prepare_models(root)
output.write_text(json.dumps(result,indent=2)+'\n')
"""
SELECTION = Path('/content/corpus_model_selection.json')
subprocess.run([PYTHON,'-c',selection_code,str(MODEL_PATH),str(SELECTION)],cwd=ROOT,check=True)
selection = json.loads(SELECTION.read_text())
manifest_root = STORE/'model_manifests'
manifest_root.mkdir(exist_ok=True)
model_manifest = manifest_root/(selection['revision']+'.json')
if model_manifest.exists():
    if json.loads(model_manifest.read_text()) != selection:
        raise RuntimeError('Saved model manifest differs; inspect it before continuing.')
else:
    model_manifest.write_text(json.dumps(selection,indent=2)+'\n')
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
(manifest_root/f'requirements_{stamp}.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
(manifest_root/f'repository_commit_{stamp}.txt').write_text(subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True))


In [ ]:
configure_code = r"""
import sys,yaml
from pathlib import Path
path=Path(sys.argv[1]);config=yaml.safe_load(path.read_text())
config['processing']['ocr'].update(backend='surya',model_path=sys.argv[2],revision=sys.argv[3])
config['processing'].update(max_pages=int(sys.argv[4]),max_documents=1,ocr_mode='all')
config['download']['max_documents']=1
path.write_text(yaml.safe_dump(config,sort_keys=False,allow_unicode=True))
"""
subprocess.run([PYTHON,'-c',configure_code,str(CONFIG),str(MODEL_PATH),selection['revision'],str(MAX_PAGES)],cwd=ROOT,check=True)

def execute(*args):
    # Capture only the final JSON; visible progress goes to stderr.
    result = subprocess.run([PYTHON,'-B','-m','pipeline.workflow',*args],cwd=ROOT,text=True,stdout=subprocess.PIPE)
    print(result.stdout)
    result.check_returncode()
    return json.loads(result.stdout)

def selection_args():
    return [part for doc in DOC_IDS for part in ['--doc-id',doc]]


## Download preview, then the bounded download

This selects an existing public-source catalog record. To discover more resources later,
use the persistent-catalog discovery command in the guide. Downloads respect configured
source approval, robots policy, host restrictions and transfer limits.


In [ ]:
execute('collect','--config',str(CONFIG),*selection_args(),'--dry-run')


In [ ]:
download_report = execute('collect','--config',str(CONFIG),*selection_args(),'--progress','plain')


## Process selected pages

Each page is saved to Drive immediately. `predicted` means generation completed;
it does not mean the text is accurate. Failed/incomplete pages and warnings remain visible.
Rerun this cell to verify existing pages and process remaining selected pages within the
batch cap. To expand, change `PAGE_NUMBERS`/`MAX_PAGES`, rerun the configuration cell and
this cell. Keep the same store. For retrying a failed OCR output, use a new output root/run
variant as described in the guide; existing artifacts are immutable.


In [ ]:
page_args = [] if PAGE_NUMBERS is None else [part for page in PAGE_NUMBERS for part in ['--page-number',str(page)]]
processing_report = execute('process','--config',str(CONFIG),*selection_args(),*page_args,'--progress','plain')
execute('status','--config',str(CONFIG))


## Prepare the manual review packet

A new review folder contains `index.html`, source page images, editable `texts/*.txt`,
`review.csv` and provenance. Download the complete packet from Drive and open `index.html`
locally, or view individual images and text files in Drive. The index shows the initial
prediction; edited text files are the authoritative correction drafts.

Have one teammate correct the text and a different teammate verify the full source,
equations and reading order. Do not approve unreadable material. Approval/export commands
are in the guide and are deliberately not executed by this notebook.


In [ ]:
review_stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
REVIEW = STORE/'corpus/clean/reviews'/f'surya_{review_stamp}'
execute('prepare-review','--config',str(CONFIG),'--output',str(REVIEW),
        '--config-hash',processing_report['processing_config_sha256'])
print('Review packet:', REVIEW)


## Review coverage and quality report

This offline report identifies pending review, OCR failures, invalid approvals and split/duplicate blockers.
All newly prepared pages are pending: that is expected. Nothing is approved by this cell.
After editing and manually approving source-checked corrections, rerun this cell to save a new report snapshot.
Correction distances measure editing burden, not independent OCR accuracy. Read `docs/CORPUS_QUALITY_GUIDE.md`.


In [ ]:
quality_stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
QUALITY = STORE/'corpus/logs/quality'/f'surya_{quality_stamp}'
quality_report = execute('quality','--packet',str(REVIEW),'--output',str(QUALITY))
print('Read the summary on Drive:', QUALITY/'report.md')


## Optional: continue acquisition while review is deferred

You can collect provisional OCR text now without retyping every page. No accuracy percentage is certified.
Set `RUN_MORE = True` below to process the selected document in bounded cycles. This selects the whole PDF,
including pages beyond the earlier two-page trial; completed artifacts are verified and reused.
The saved text remains unreviewed, with warnings and `training_approved=false`. It is not a reviewed training release.
Start with ten new pages and read `docs/BATCH_COLLECTION_GUIDE.md` before increasing the caps.


In [ ]:
RUN_MORE = False
batch_report = None
if RUN_MORE:
    batch_report = execute('run','--config',str(CONFIG),*selection_args(),
                           '--max-batches','5','--max-download-jobs','1',
                           '--max-new-pages','10','--progress','plain')
else:
    print('Set RUN_MORE=True to run the bounded provisional collection batch.')


In [ ]:
if batch_report and batch_report.get('processed_bundles'):
    provisional_stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    PROVISIONAL = STORE/'corpus/clean/provisional'/f'surya_{provisional_stamp}'
    execute('provisional','--run-report',batch_report['report_path'],'--output',str(PROVISIONAL))
    print('Unreviewed text snapshot:', PROVISIONAL/'provisional_text.jsonl')


## Optional: daily training release

Build filtered experimental text candidates from the provisional snapshot. No model training or submission runs here.
Set `BUILD_DAILY=True`. For the first release leave `LAST_CONSUMED_RELEASE=None`. Later, use the release actually
consumed by the checkpoint you are continuing from, not automatically the latest folder.
Use `train.jsonl` for a fresh base and `added_train.jsonl` for a matching continual checkpoint.
Incomplete/warning-bearing OCR is quarantined. Read `docs/DAILY_TRAINING_RELEASE_GUIDE.md`.


In [ ]:
BUILD_DAILY = False
LAST_CONSUMED_RELEASE = None
if BUILD_DAILY:
    if 'PROVISIONAL' not in globals() or not (PROVISIONAL/'manifest.json').is_file():
        raise RuntimeError('Create the provisional snapshot first, or set PROVISIONAL to its actual folder.')
    daily_stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    DAILY = STORE/'datasets/pretraining'/f'daily_{daily_stamp}'
    daily_args = ['--provisional',str(PROVISIONAL),'--output',str(DAILY)]
    if LAST_CONSUMED_RELEASE is not None: daily_args += ['--previous',str(LAST_CONSUMED_RELEASE)]
    daily_report = execute('daily-release',*daily_args)
    execute('verify-daily-release','--root',str(DAILY))
    print('Training handoff:', DAILY/'handoff.json')
    print('Release counts:', daily_report['counts'])
else:
    print('Set BUILD_DAILY=True when you want to freeze training candidates.')


## Separate theory and source-backed MCQs

The daily release contains mixed page text. Set `PREPARE_TASKS=True` to prepare numbered public-source text
and editable line annotations. Open `pages.txt` beside the original pages, then edit `annotations.jsonl`.
Select theory ranges, MCQ question/choice ranges, and exclusions. MCQ answers need source evidence and a
recorded source check; unanswered items remain candidates. Do not use any hidden benchmark material.
Read `docs/THEORY_AND_MCQ_GUIDE.md`. The source text is selected by line number, so no Sinhala retyping is needed.


In [ ]:
PREPARE_TASKS = False
if PREPARE_TASKS:
    if 'DAILY' not in globals() or not (DAILY/'manifest.json').is_file():
        raise RuntimeError('Set DAILY to the completed daily release folder first.')
    task_stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    TASK_PACKET = STORE/'reviews/tasks'/f'tasks_{task_stamp}'
    execute('prepare-tasks','--release',str(DAILY),'--output',str(TASK_PACKET))
    print('Read:', TASK_PACKET/'pages.txt')
    print('Edit source line annotations:', TASK_PACKET/'annotations.jsonl')
else:
    print('Set PREPARE_TASKS=True to start source classification.')


## Optional: match a public paper with its answer key

Use a newly prepared task packet for the question paper, then fill a copy of
`configs/paper_pairs.example.yaml` from the public paper and marking scheme.
The current Biology guide is not a paper/key pilot. Read `docs/MCQ_ANSWER_MATCHING_GUIDE.md`.
Set the config path below; optional key bundles supply numeric key pages omitted by prose filtering.
Matching makes proposals only. Review source question/choices and answers before export.
Never supply hidden competition questions or answers.


In [ ]:
MATCH_ANSWERS = False
PAIR_CONFIG = STORE/'configs/paper_pairs.yaml'
KEY_BUNDLES = []  # Actual processed page folders, if numeric keys are absent from TASK_PACKET.
if MATCH_ANSWERS:
    if 'TASK_PACKET' not in globals() or not (TASK_PACKET/'manifest.json').is_file():
        raise RuntimeError('Prepare a fresh task packet for the public question paper first.')
    if not PAIR_CONFIG.is_file():
        raise RuntimeError('Fill paper_pairs.example.yaml from the originals and save it as PAIR_CONFIG.')
    match_stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    MATCHED_PACKET = STORE/'reviews/tasks'/f'matched_{match_stamp}'
    match_args = ['--packet',str(TASK_PACKET),'--pairs',str(PAIR_CONFIG),'--output',str(MATCHED_PACKET)]
    if KEY_BUNDLES:
        if 'DAILY' not in globals():
            raise RuntimeError('Set DAILY to the originating daily release for key split checks.')
        match_args += ['--release',str(DAILY)]
        for bundle in KEY_BUNDLES: match_args += ['--key-bundle',str(bundle)]
    match_report = execute('match-paper-answers',*match_args)
    TASK_PACKET = MATCHED_PACKET
    print('Review matches:', TASK_PACKET/'matching_report.json')
    print('Check source questions/choices/answers:', TASK_PACKET/'annotations.jsonl')
else:
    print('Fill the public paper/key configuration before setting MATCH_ANSWERS=True.')


In [ ]:
EXPORT_TASKS = False
if EXPORT_TASKS:
    if 'TASK_PACKET' not in globals() or not (TASK_PACKET/'manifest.json').is_file():
        raise RuntimeError('Set TASK_PACKET to the annotated task packet folder first.')
    task_stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    TASK_RELEASE = STORE/'datasets/tasks'/f'tasks_{task_stamp}'
    task_report = execute('export-tasks','--packet',str(TASK_PACKET),'--output',str(TASK_RELEASE))
    execute('verify-task-release','--root',str(TASK_RELEASE))
    print('Theory and MCQ release:', TASK_RELEASE)
    print('Counts:', task_report['counts'])
    print('Unclassified pages and unchecked answers are not training examples.')
else:
    print('Edit the annotations before setting EXPORT_TASKS=True.')
